# 01 · Datos: estadísticas NFL + mi liga de ESPN

**Objetivo del proyecto:** predecir los puntos de fantasy de los jugadores de mi liga de ESPN.

En este notebook:
1. Descargo las estadísticas semanales de jugadores de las temporadas 2023–2026 con [`nflreadpy`](https://github.com/nflverse/nflreadpy).
2. Me conecto a mi liga de ESPN con [`espn-api`](https://github.com/cwendt94/espn-api), y verifico que sus reglas coinciden con `config/scoring.yaml` y con `fantasy_points_ppr` de nflverse.
3. Muestro mi roster con las proyecciones de ESPN y lo cruzo con los datos de nflverse mediante IDs.

> Las credenciales se leen desde `.env` (no versionado). Este notebook nunca imprime sus valores.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import polars as pl
import yaml
import nflreadpy as nfl
from dotenv import load_dotenv
from espn_api.football import League

# Rutas relativas a la raíz del proyecto (el notebook vive en notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
CONFIG = ROOT / "config"
DATA_RAW.mkdir(parents=True, exist_ok=True)
CONFIG.mkdir(exist_ok=True)

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(15)

In [ ]:
load_dotenv(ROOT / ".env")

REQUIRED = ["ESPN_LEAGUE_ID", "ESPN_S2", "ESPN_SWID", "ESPN_TEAM_ID"]
missing = [k for k in REQUIRED if not os.getenv(k)]
for k in REQUIRED:
    print(f"{'✓' if k not in missing else '✗'} {k} {'cargada' if k not in missing else 'FALTA'}")
if missing:
    raise EnvironmentError(f"Faltan variables en .env: {missing}. Usa .env.example como plantilla.")

## 2. Estadísticas semanales 2023–2026 (nflverse)

`load_player_stats` devuelve una fila por jugador-semana, con más de 100 columnas de estadísticas y dos columnas de puntos ya calculadas: `fantasy_points` (estándar) y `fantasy_points_ppr`.

Guardo el resultado en parquet como caché. Pon `REFRESH = True` para volver a descargar; hazlo cada semana durante la temporada, porque 2026 está en curso.

In [ ]:
SEASONS = [2023, 2024, 2025, 2026]
REFRESH = False
stats_path = DATA_RAW / "player_stats_weekly_2023_2026.parquet"

if stats_path.exists() and not REFRESH:
    stats = pl.read_parquet(stats_path)
    print(f"Cargado desde caché: {stats_path.relative_to(ROOT)}")
else:
    stats = nfl.load_player_stats(SEASONS, summary_level="week")
    stats.write_parquet(stats_path)
    print(f"Descargado y guardado en: {stats_path.relative_to(ROOT)}")

print(f"{stats.height:,} filas × {stats.width} columnas")

### Cobertura por temporada

2026 está en curso, así que solo tiene las semanas jugadas hasta hoy.

In [ ]:
print(f"Temporada/semana actual según nflverse: {nfl.get_current_season()} / semana {nfl.get_current_week()}")

(stats
 .group_by("season", "season_type")
 .agg(pl.col("week").min().alias("semana_min"),
      pl.col("week").max().alias("semana_max"),
      pl.col("player_id").n_unique().alias("jugadores"),
      pl.len().alias("filas"))
 .sort("season", pl.col("season_type").replace_strict({"REG": 0, "POST": 1}, default=2)))

### Posiciones fantasy

Me concentro en QB, RB, WR y TE, que son las posiciones con estadísticas individuales. K y D/ST se tratarán aparte.

In [ ]:
FANTASY_POS = ["QB", "RB", "WR", "TE"]

(stats
 .filter(pl.col("season_type") == "REG", pl.col("position").is_in(FANTASY_POS))
 .group_by("position")
 .agg(pl.len().alias("filas"),
      pl.col("fantasy_points_ppr").mean().round(2).alias("ppr_media"),
      pl.col("fantasy_points_ppr").median().round(2).alias("ppr_mediana"),
      pl.col("fantasy_points_ppr").max().round(2).alias("ppr_max"))
 .sort("ppr_media", descending=True))

### Valores nulos en las columnas clave

In [ ]:
key_cols = ["player_id", "player_display_name", "position", "team", "opponent_team",
            "targets", "receptions", "carries", "attempts", "fantasy_points", "fantasy_points_ppr"]
(stats.select(key_cols).null_count()
 .transpose(include_header=True, header_name="columna", column_names=["nulos"])
 .with_columns((pl.col("nulos") / stats.height * 100).round(2).alias("pct")))

## 3. Reglas de puntuación (`config/scoring.yaml`)

La liga es **PPR completo con el scoring estándar de ESPN**. Las reglas completas están en `config/scoring.yaml`, copiadas de la pantalla *Scoring* de la liga. Hay reglas para jugadores ofensivos, K y D/ST, y los rangos de puntos y yardas permitidas están definidos con `min`/`max`, listos para calcular.

- **QB/RB/WR/TE:** el objetivo es `fantasy_points_ppr`, que ya viene calculado en nflverse.
- **K y D/ST:** nflverse no trae sus puntos de fantasy, así que los calcularé con este YAML en el notebook 02.

In [ ]:
SCORING = yaml.safe_load((CONFIG / "scoring.yaml").read_text(encoding="utf-8"))
TARGET = SCORING["target_offense"]

def flatten_rules(cfg):
    """Todas las reglas del YAML en una lista plana (offense + kicking + dst)."""
    dst = cfg["dst"]
    return ([r for group in cfg["offense"].values() for r in group]
            + cfg["kicking"] + dst["events"] + dst["points_allowed"] + dst["yards_allowed"])

# Algunas reglas aparecen dos veces (p. ej. KRTD en misc y en D/ST): deben valer lo mismo
yaml_pts, yaml_labels = {}, {}
for r in flatten_rules(SCORING):
    prev = yaml_pts.setdefault(r["stat_id"], float(r["points"]))
    if prev != float(r["points"]):
        raise ValueError(f"stat_id {r['stat_id']} tiene dos valores distintos en scoring.yaml")
    yaml_labels[r["stat_id"]] = r.get("label", r["abbr"])

print(f"{len(yaml_pts)} reglas únicas en config/scoring.yaml · formato {SCORING['format']} · objetivo: {TARGET}")

### ¿`fantasy_points_ppr` de nflverse usa las mismas reglas?

nflverse calcula `fantasy_points_ppr` con una fórmula fija. Si coincide con las reglas ofensivas de la liga, puedo usar esa columna directamente sin recalcular nada.

In [ ]:
NFLVERSE_PPR = {  # statId ESPN → puntos en la fórmula de fantasy_points_ppr de nflverse
    3: 0.04, 4: 4, 20: -2, 19: 2,         # pase
    24: 0.1, 25: 6, 26: 2,                # carrera
    42: 0.1, 53: 1, 43: 6, 44: 2,         # recepción
    72: -2, 101: 6, 102: 6,               # fumble perdido, TD de retorno (special_teams_tds)
}
offense = [r for group in SCORING["offense"].values() for r in group]
check = (pl.DataFrame([{"stat_id": r["stat_id"], "abbr": r["abbr"], "liga": float(r["points"]),
                        "nflverse": NFLVERSE_PPR.get(r["stat_id"])} for r in offense],
                      schema_overrides={"nflverse": pl.Float64})
         .with_columns(pl.when(pl.col("nflverse").is_null()).then(pl.lit("no la cuenta"))
                         .when((pl.col("liga") - pl.col("nflverse")).abs() < 1e-9).then(pl.lit("✓"))
                         .otherwise(pl.lit("✗ distinta")).alias("estado")))

if (check["estado"] == "✗ distinta").any():
    raise ValueError("Alguna regla ofensiva difiere de nflverse: hay que recalcular los puntos en vez de usar fantasy_points_ppr")
print("✓ Todas las reglas que nflverse cuenta coinciden con la liga.")
print("  Las marcadas 'no la cuenta' son jugadas rarísimas para un jugador ofensivo (TD tras recuperar un fumble, retorno de 2 pts, safety de 1 pt).")
check

## 4. Conexión a mi liga de ESPN

Para una liga privada hacen falta las cookies `espn_s2` y `SWID`. Al conectar solo imprimo metadatos no sensibles.

In [ ]:
league = League(
    league_id=int(os.environ["ESPN_LEAGUE_ID"]),
    year=2026,
    espn_s2=os.environ["ESPN_S2"],
    swid=os.environ["ESPN_SWID"],
)
MY_TEAM_ID = int(os.environ["ESPN_TEAM_ID"])

s = league.settings
print(f"Liga:            {s.name}")
print(f"Equipos:         {s.team_count}")
print(f"Semana actual:   {league.current_week}")
print(f"Tipo de puntos:  {s.scoring_type}")
print(f"Slots titulares: { {k: v for k, v in s.position_slot_counts.items() if v} }")

### Verificación: configuración de la liga vs `config/scoring.yaml`

Comparo regla por regla (`stat_id`). Si la liga cambia sus reglas, o si el YAML tiene un error de transcripción, el notebook se detiene aquí y no se usan puntos equivocados. Las reglas que valen 0 y no aparecen en ESPN cuentan como coincidentes.

In [ ]:
league_pts = {int(it["id"]): float(it["points"] or 0) for it in s.scoring_format}
league_labels = {int(it["id"]): it["label"] for it in s.scoring_format}

diff = (pl.DataFrame([
            {"stat_id": sid, "regla": league_labels.get(sid) or yaml_labels.get(sid),
             "yaml": yaml_pts.get(sid), "liga": league_pts.get(sid)}
            for sid in sorted(yaml_pts.keys() | league_pts.keys())],
            schema_overrides={"yaml": pl.Float64, "liga": pl.Float64})
        .filter((pl.col("yaml").fill_null(0) - pl.col("liga").fill_null(0)).abs() > 1e-9))

if diff.height:
    display(diff)
    raise ValueError(f"{diff.height} reglas difieren entre la liga y config/scoring.yaml (tabla arriba)")
print(f"✓ config/scoring.yaml coincide con la configuración de la liga ({len(league_pts)} reglas en ESPN)")

## 5. Mi roster con proyecciones de ESPN

- **Proyección semanal:** sale del box score de la semana actual (`projected_points` por jugador), con el mismo número que ves en la app.
- **Proyección de temporada:** `projected_total_points` del jugador.
- **Slot:** si el jugador es titular o está en banca (BE) o en IR.

In [ ]:
my_team = next((t for t in league.teams if t.team_id == MY_TEAM_ID), None)
if my_team is None:
    raise ValueError(f"ESPN_TEAM_ID={MY_TEAM_ID} no existe. IDs válidos: {[t.team_id for t in league.teams]}")

print(f"Mi equipo: {my_team.team_name}  ({my_team.wins}-{my_team.losses})")

In [ ]:
WEEK = league.current_week
box = next((b for b in league.box_scores(WEEK)
            if MY_TEAM_ID in (getattr(b.home_team, "team_id", None), getattr(b.away_team, "team_id", None))),
           None)

if box is not None:
    is_home = box.home_team.team_id == MY_TEAM_ID
    lineup = box.home_lineup if is_home else box.away_lineup
    rival = box.away_team if is_home else box.home_team
    print(f"Semana {WEEK} vs {getattr(rival, 'team_name', 'BYE')}")
    weekly = {p.playerId: p for p in lineup}
else:
    print(f"Sin enfrentamiento en la semana {WEEK}; solo muestro las proyecciones de temporada.")
    weekly = {}

SLOT_ORDER = {"QB": 0, "RB": 1, "WR": 2, "TE": 3, "RB/WR/TE": 4, "OP": 5, "D/ST": 6, "K": 7, "BE": 8, "IR": 9}
PROY_SEM = f"proy_sem{WEEK}"

roster = pl.DataFrame([
    {
        "espn_id": p.playerId,
        "jugador": p.name,
        "pos": p.position,
        "equipo_nfl": p.proTeam,
        "slot": getattr(weekly.get(p.playerId), "slot_position", p.lineupSlot),
        "rival_nfl": getattr(weekly.get(p.playerId), "pro_opponent", None),
        "estado": p.injuryStatus,
        PROY_SEM: getattr(weekly.get(p.playerId), "projected_points", None),
        "proy_temporada": p.projected_total_points,
        "pts_temporada": p.total_points,
    }
    for p in my_team.roster
])

roster = (roster
          .with_columns(pl.col("slot").replace_strict(SLOT_ORDER, default=99).alias("_orden"))
          .sort(["_orden", PROY_SEM], descending=[False, True], nulls_last=True)
          .drop("_orden"))
roster

In [ ]:
starters = roster.filter(~pl.col("slot").is_in(["BE", "IR"]))
print(f"Proyección ESPN de titulares, semana {WEEK}: {starters[PROY_SEM].sum():.1f} pts")

## 6. Cruce ESPN ↔ nflverse por ID

Para entrenar con el historial de cada jugador necesito su `gsis_id` (el `player_id` de nflverse). `load_ff_playerids()` trae el mapeo `espn_id → gsis_id`, que es más robusto que cruzar por nombre (sufijos Jr., apodos, tildes).
Los D/ST no tienen `gsis_id` y quedan fuera por diseño.

In [ ]:
ids = (nfl.load_ff_playerids()
       .select("espn_id", "gsis_id")
       .drop_nulls()
       .unique("espn_id"))

hist = (stats
        .filter(pl.col("season_type") == "REG")
        .group_by("player_id")
        .agg(pl.len().alias("partidos_2023_26"),
             pl.col(TARGET).mean().round(2).alias("ppr_media")))

roster_ids = (roster
              .with_columns(pl.col("espn_id").cast(pl.Int64))
              .join(ids, on="espn_id", how="left")
              .join(hist, left_on="gsis_id", right_on="player_id", how="left"))

no_match = roster_ids.filter(pl.col("gsis_id").is_null() & (pl.col("pos") != "D/ST"))
print(f"Jugadores con gsis_id: {roster_ids['gsis_id'].is_not_null().sum()}/{roster_ids.height} "
      f"(sin mapeo, excluyendo D/ST: {no_match.height})")
roster_ids.select("jugador", "pos", "slot", "gsis_id", PROY_SEM, "partidos_2023_26", "ppr_media")

## Resumen y siguientes pasos

- ✅ Estadísticas semanales 2023–2026 en `data/raw/` (con caché).
- ✅ `config/scoring.yaml` verificado contra la liga y contra la fórmula de nflverse → objetivo `fantasy_points_ppr` para QB/RB/WR/TE.
- ✅ Roster con proyecciones semanales y de temporada de ESPN, cruzado con nflverse por ID.

**Siguiente: `02_features.ipynb`.** Calcularé los puntos de K y D/ST con `config/scoring.yaml` y construiré features rodantes (targets, carries, snap share, rival) sin fuga de información hacia el futuro, y usaré las proyecciones de ESPN como baseline contra el que comparar el modelo.